<picture>
  <source media="(prefers-color-scheme: dark)" srcset="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-dark.svg">
  <img alt="Meep²" src="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-light.svg" width="239" height="80">
</picture>

# 01 · LoRA and QLoRA fine-tuning with meepsquared

> **Licence: Apache-2.0.** Copyright Alejandro Martínez Corriá and the Thinkube contributors.
> This notebook, [meepsquared](https://github.com/thinkube/meepsquared) and every package meepsquared installs are permissively licensed:
> you can use, change and ship what you build with them, commercially or not. See [Licensing](https://thinkube.org/meepsquared/licensing.html).

**Where it runs.** On any Linux machine with an NVIDIA GPU of the Ampere generation or later and Meep² installed ([Install](https://github.com/thinkube/meepsquared#install)). The model and the data come from the Hugging Face Hub. Run the cells from top to bottom.

A few cells are marked **Thinkube only**. [Thinkube](https://thinkube.org) is the open-source AI platform Meep² is developed on; on a Thinkube cluster these cells read the model from the cluster's copy, record the run and serve the result. Elsewhere they do nothing, so run every cell as it comes.

The outputs saved in this notebook come from a run on a Thinkube cluster, on an NVIDIA DGX Spark, so they show the cluster's model paths and its run links.

## What you will do

Fine-tune a chat model on conversations, and see what changed. The notebook goes in order: look at the data, plan the training job, train, ask the base model and the trained model the same two questions, save the adapter. On Thinkube it also follows the run in Thinkube Experiments and registers the model so the cluster can serve it.

### The two ideas behind it

**LoRA** (low-rank adaptation) fine-tunes a model without changing its weights. Next to each large weight matrix W it adds two small ones, A and B, whose product is a correction of low *rank* (16 here). The layer computes `x·W + x·A·B`; only A and B learn, W stays frozen. An 8B model has 8 billion frozen numbers and about 40 million trainable ones, so the training needs a fraction of the memory and optimizer state of a full fine-tune, and the result, the **adapter**, is a few tens of megabytes that load onto the base model.

**QLoRA** does the same with the frozen weights held in 4 bits instead of 16 (`METHOD = "qlora"`). The 16 GB of an 8B model become about 5, at the cost of some speed for dequantizing on the fly; the adapters stay in 16 bits. It is how an 8B model trains on a 12 GB GPU, or a 27B model on a 24 GB one.

### What meepsquared adds

You describe the training as a **training job**: the model, the data and the method. meepsquared does the rest in *auto mode*: it measures the model, the batch and your GPU, and decides how to train on them. It chooses padding-free batches (conversations packed without wasted padding tokens), fused kernels for the adapters and the MLP, compiled layers where they pay, gradient checkpointing *measured* for your batch (so only as many layers are recomputed as the free memory requires), the batch size and the learning rate. Before it trains, it prints a **plan**: one line for each decision, with its reason. Underneath, it trains with TRL's own `SFTTrainer`.

The defaults run 30 steps on no_robots, a few minutes on one GPU: enough to see the loss fall and the answers change. For a real fine-tune, point `DATASET` at your conversations and set `EPOCHS`. Llama 3.1 8B Instruct by default; any dense chat model works (`Qwen/Qwen3.5-9B`, `google/gemma-4-E4B-it`).

Needs `meepsquared` and `pandas`; on Thinkube, the `meepsquared` kernel has them.

## Settings

Every choice of the notebook is a name in this cell, so you can rerun the notebook with other values without touching the cells below, and a notebook job can override any of them with its parameters.

In [2]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "huggingface"; "thinkube" only on a Thinkube cluster
MODEL = PARAMETERS.get("MODEL", "Qwen/Qwen3.5-9B")
METHOD = PARAMETERS.get("METHOD", "lora")                    # "lora" (BF16 base) or "qlora" (4-bit base)
DATASET = PARAMETERS.get("DATASET", "HuggingFaceH4/no_robots")  # a Hub dataset, a Parquet file or a JSONL file of conversations
ROWS = PARAMETERS.get("ROWS", 480)                           # conversations to use; None for the whole dataset
EPOCHS = PARAMETERS.get("EPOCHS", None)                      # epochs over ROWS; None trains STEPS steps instead
STEPS = PARAMETERS.get("STEPS", 30)
# Auto mode chooses these; set one to choose it yourself, and the plan shows meepsquared's choice beside yours
RANK = PARAMETERS.get("RANK", None)                          # LoRA rank, with alpha equal to it; auto: 16
LEARNING_RATE = PARAMETERS.get("LEARNING_RATE", None)        # auto: 1e-4 up to 13B parameters; 2e-4 is fast, 5e-5 careful
BATCH = PARAMETERS.get("BATCH", None)                        # conversations per update of the weights; auto: 8 up to 13B parameters
MAX_LENGTH = PARAMETERS.get("MAX_LENGTH", None)              # tokens per conversation, longer ones are cut; auto: 1024
OUTPUT_DIR = PARAMETERS.get("OUTPUT_DIR", "out/01-lora-sft")  # where the adapter is saved
SAVE_MERGED = PARAMETERS.get("SAVE_MERGED", False)           # also save the merged 16-bit model (the base model's size)
# Thinkube only: ignored elsewhere
EXPERIMENT = PARAMETERS.get("EXPERIMENT", "meepsquared")
RUN_NAME = PARAMETERS.get("RUN_NAME", f"01-{METHOD}-{MODEL.split('/')[-1]}")
REGISTER = PARAMETERS.get("REGISTER", False)                 # merge and register the model in the cluster's catalogue
FINETUNED_NAME = PARAMETERS.get("FINETUNED_NAME", f"{MODEL.split('/')[-1].lower()}-no-robots")

## Where the model comes from

A fine-tuning starts from a **base model**: a model somebody already trained on a great deal of text, whose behaviour you will adjust with your own data. The base model is the largest thing a training handles, 8 to 70 GB of weights for the models in these notebooks, so where it comes from matters.

`MODEL_SOURCE` chooses:

* `"huggingface"`: the model is downloaded from the Hugging Face Hub by its id, once, into the Hub's cache on this machine. A *gated* model, such as Llama, needs its licence accepted on its Hub page and the token of your account in the `HF_TOKEN` variable.
* `"thinkube"`, only on a [Thinkube](https://thinkube.org) cluster: the model comes from the cluster's **model mirror**: a copy made once into the cluster's shared storage, which every GPU node reads. Nothing is downloaded by a training, a gated model needs no token here, and a 70 GB model loads in about a minute. A model is mirrored in Thinkube Control (AI > Models). `thinkube_models.mirrored_model_path` finds its folder, and stops with an error naming the model if it is not mirrored, before any time is spent.

In [3]:
import os
from pathlib import Path

if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    MODEL_PATH = tkm.mirrored_model_path(MODEL)
    print(f"{MODEL}: {MODEL_PATH}")
elif MODEL_SOURCE == "huggingface":
    MODEL_PATH = MODEL
    print(f"{MODEL}: from the Hugging Face Hub")
else:
    raise ValueError(f"MODEL_SOURCE must be 'thinkube' or 'huggingface', not {MODEL_SOURCE!r}")

Qwen/Qwen3.5-9B: /home/thinkube/thinkube/mlflow/artifacts/1/3fff0933294740a6ae61724295182e1a/artifacts/model


## The data

Supervised fine-tuning learns from examples of the behaviour you want. Here the examples are conversations. Each one is a list of turns, `{"role": "user" | "assistant", "content": ...}`, the shape the model's **chat template** turns into the exact text the model sees: special tokens mark where each turn begins and ends, so the model learns where an answer starts. Any dataset in that shape works:

* a Hugging Face dataset id with a `messages` column;
* a Parquet file, or a JSONL file with one conversation per line, with the same `messages` field.

`ROWS` takes the first conversations of the dataset, `None` all of them. The default, `HuggingFaceH4/no_robots`, is 10,000 conversations written by people: varied, clean and small, a good first dataset. A real fine-tune uses your own conversations. A few hundred good ones already change a model's tone; a few thousand change what it knows how to do.

The cell prints the first conversation, so you can see the shape the model will learn from.

In [4]:
import json

from datasets import Dataset, load_dataset


def conversations(source, rows=None):
    """Conversations as {"messages": [...]}, from a Hub dataset, a Parquet file or a JSONL file."""
    if source.endswith(".jsonl"):
        data = Dataset.from_list([json.loads(line) for line in open(source) if line.strip()])
    elif source.endswith(".parquet"):
        data = load_dataset("parquet", data_files=source, split="train")
    else:
        data = load_dataset(source, split="train")
    if "messages" not in data.column_names:
        raise ValueError(f"{source} has no 'messages' column; its columns are {data.column_names}")
    if rows is not None:
        if len(data) < rows:
            raise ValueError(f"{source} has {len(data)} conversations; ROWS asks for {rows}")
        data = data.select(range(rows))
    return data.map(lambda r: {"messages": [{"role": m["role"], "content": m["content"]} for m in r["messages"]]},
                    remove_columns=data.column_names)


train_data = conversations(DATASET, ROWS)
print(f"{len(train_data)} conversations; the first:")
for turn in train_data[0]["messages"]:
    print(f"  {turn['role']}: {turn['content'][:200]!r}")

README.md:   0%|          | 0.00/5.61k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  571kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

480 conversations; the first:
  user: 'Please summarize the goals for scientists in this text:\n\nWithin three days, the intertwined cup nest of grasses was complete, featuring a canopy of overhanging grasses to conceal it. And decades later'
  assistant: 'Scientists are studying nests hoping to learn about transitional habitats that could help restore the shoreline of San Francisco Bay.'


## Thinkube only: the training run in Thinkube Experiments

*Elsewhere the cell only sets `REPORT_TO` to an empty list; run it anyway, the training job below uses it.*

A training is an experiment: settings in, a loss curve and a model out. Keeping that record is what lets you compare two runs later, or trace a served model back to the data and settings that made it. On Thinkube the record is **Thinkube Experiments** (MLflow). The cell opens a *run* there, before the training job is described: the job's `report_to=REPORT_TO` then sends every step's loss to that run as it trains, so the curve is on the experiments page while the run goes. Elsewhere the metrics are in the trainer's log below, and `REPORT_TO` is empty.

In [5]:
RUN_ID = None
if MODEL_SOURCE == "thinkube":
    import mlflow
    import thinkube_models as tkm

    os.environ["MLFLOW_TRACKING_TOKEN"] = tkm.get_mlflow_token()
    mlflow.set_tracking_uri("http://mlflow.mlflow.svc.cluster.local:5000")
    experiment = mlflow.get_experiment_by_name(EXPERIMENT)
    experiment_id = experiment.experiment_id if experiment else mlflow.create_experiment(EXPERIMENT)
    run = mlflow.start_run(experiment_id=experiment_id, run_name=RUN_NAME)
    RUN_ID = run.info.run_id
    REPORT_TO = ["mlflow"]
    print(f"run {RUN_ID} in experiment {EXPERIMENT!r}: open Thinkube Experiments to follow it")
else:
    REPORT_TO = []

run 86ba073a98db4a99836d71c1a3b4608a in experiment 'meepsquared': open Thinkube Experiments to follow it


## The training job and its plan

`m2.TrainingJob` describes the training: the model, the data, the method, and how long. `METHOD` is `"lora"` (the frozen weights in BF16) or `"qlora"` (in 4 bits, through bitsandbytes). The settings left as `None` go to auto mode. The ones you set go in an `m2.Recipe` (the training choices), and the plan marks them as yours, next to what meepsquared would have chosen.

`job.plan()` does the work that comes before training. It loads the model, adds the LoRA matrices on the seven projections of every layer (the four of the attention, `q_proj`, `k_proj`, `v_proj`, `o_proj`, and the three of the MLP, `gate_proj`, `up_proj`, `down_proj`), measures the memory one batch needs on your GPU, and builds TRL's trainer. Nothing is trained yet. Read the plan: it names the GPU and its free memory, how the frozen weights are held, which kernels run, how many layers are checkpointed and why, the batch, the learning rate and the schedule. LoRA's alpha, the scale of the correction, equals the rank unless you set it.

`job.trainer()` is the TRL `SFTTrainer` that `job.train()` will run. The cell takes the model and the tokenizer from it, to ask the model questions and to save it below.

In [6]:
import meepsquared as m2

job = m2.TrainingJob(
    model=MODEL_PATH, data=train_data, method=METHOD, output=OUTPUT_DIR,
    steps=None if EPOCHS else STEPS, epochs=EPOCHS, report_to=REPORT_TO,
    recipe=m2.Recipe(rank=RANK, learning_rate=LEARNING_RATE, effective_batch=BATCH, max_length=MAX_LENGTH),
)
print(job.plan())  # loads the model, adds LoRA, measures the memory; nothing is trained yet

trainer = job.trainer()  # TRL's SFTTrainer, set up as job.train() will run it
model, tokenizer = trainer.model, trainer.processing_class

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

╭────────────╮      Meep² 0.1.0: fast LoRA and QLoRA on Transformers, PEFT and TRL
│ meep meep! │      GPU: NVIDIA GB10 (sm_121), allocator: expandable segments. PyTorch 2.11.0a0. Transformers 5.20.2. TRL 1.16.0
╰──┬─────────╯      Model: qwen3_5_text, bfloat16. Liger: on. Padding-free: on
   ╵ ∘ ○  ·  ·  ·   Apache-2.0: https://github.com/thinkube/meepsquared


[RANK 0] You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 annihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.


Tokenizing train dataset:   0%|          | 0/480 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/480 [00:00<?, ? examples/s]

/home/thinkube/venvs/custom/meepsquared/arm64/lib/python3.12/site-packages/meepsquared/_trl.py:258: FutureWarning: `use_liger_kernel=True` is deprecated and will be removed in v2.0.0. Use the Hub kernels instead, with `model_init_kwargs={"use_kernels": True}`.
  trainer_init(*bound.args, **bound.kwargs)


Cutting to max_length:   0%|          | 0/480 [00:00<?, ? examples/s]

/home/thinkube/venvs/custom/meepsquared/arm64/lib/python3.12/site-packages/meepsquared/_trl.py:258: FutureWarning: `use_liger_kernel=True` is deprecated and will be removed in v2.0.0. Use the Hub kernels instead, with `model_init_kwargs={"use_kernels": True}`.
  trainer_init(*bound.args, **bound.kwargs)


meepsquared plan for /home/thinkube/thinkube/mlflow/artifacts/1/3fff0933294740a6ae61724295182e1a/artifacts/model, LoRA, on NVIDIA GB10 (85 GiB free)
  loading      shard by shard to GPU 0 (the GPU shares the system's memory)
  weights      bfloat16 (method lora)
  attention    flash_attention_2 (FlashAttention 2 with packed sequences)
  batches      packed, no padding (the attention (flash_attention_2) keeps the examples apart)
  Liger        kernels and fused loss, without its SwiGLU: the fused LoRA MLP runs instead (from_pretrained's default)
  fused LoRA   on (the model's MLP has a fused kernel)
  compile      layers not compiled (FlashAttention runs outside them)
  checkpoint   0 of 32 layers (one layer for the largest of the 30 micro-batches this job trains, 3,425 tokens (8 examples, end to end); the data's 8 longest would be 7,959: 0.47 GiB keeping everything, 0.03 recomputed whole; its forward and backward 91 / 125 ms in that order; free: 85 of 122 GiB; selective recomputation n

## Before training: the questions

Two questions, asked of the base model and of the trained model. The loss curve will tell you that the training worked on the data; the answers tell you what it did to the model's behaviour, which is what you care about.

Both answers are asked after training. The base model answers with the adapter switched off: LoRA never changes the base weights, so with the adapter off the model is exactly the base model.

Two details of the helper are worth knowing. `for_inference` puts the model in generation mode (gradient checkpointing off, the layers uncompiled, the key-value cache on) and `for_training` puts it back as the plan set it; `answer` does both around one generation, so it can run at any time, between the plan and the training too. And `enable_thinking=False` asks a thinking model (Qwen 3) for a direct answer; a chat template without that switch ignores it.

In [7]:
import torch


def answer(model, tokenizer, question, max_new_tokens=160):
    """The model's answer to one user turn, with generation set up and training set back."""
    m2.MeepLanguageModel.for_inference(model)
    try:
        # enable_thinking=False keeps a thinking model's answer short; a template without the switch ignores it
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    finally:
        m2.MeepLanguageModel.for_training(model)


QUESTIONS = ["Explain in two sentences why the sky is blue.",
             "Write a haiku about a lighthouse keeper."]
print("asked after training, of the base model and of the trained model:")
for q in QUESTIONS:
    print(f"  {q}")

asked after training, of the base model and of the trained model:
  Explain in two sentences why the sky is blue.
  Write a haiku about a lighthouse keeper.


## Train

`job.train()` prints the plan again, then trains with TRL's `SFTTrainer` and every setting of the plan: the batch, split into micro-batches that fit the memory and added up by gradient accumulation; the learning rate and its schedule; the 8-bit AdamW optimizer, which keeps the optimizer state small; padding-free batches, Liger's fused layers and loss, and the checkpointed layers it measured. It returns a result: the steps, the time, the last loss and the peak memory.

The settings that matter most, all in the settings cell:

* `LEARNING_RATE`: how far each step moves the adapters. Auto mode takes `1e-4` for a model of this size; `2e-4` learns faster and risks instability, `5e-5` is careful.
* the schedule: the rate stays constant after a short warm-up, so the first updates do not wreck the freshly initialised adapters. `m2.Recipe(schedule="cosine")` makes large steps first and small ones at the end.
* `BATCH`: conversations per update of the weights. Larger is steadier. Auto mode takes the smallest batch that keeps the GPU busy, because LoRA learns less from large batches.
* `STEPS` or `EPOCHS`: how long. The defaults run a fixed number of steps to show the mechanics; a real fine-tune sets `EPOCHS` and goes through its data one to three times.

### Counting the loss on the answers only

A conversation has two kinds of tokens: the user's words and the assistant's. The model should learn to *write* the assistant's turns, not to predict the user's, so the job counts the loss on the assistant turns only, the same idea as Unsloth's `train_on_responses_only`. It does that by default for conversations. The plan's `loss on` line says how: with TRL's own assistant-only loss where TRL can mask the model's chat template, else with meepsquared's `train_on_responses_only` and the turn markers it reads from the chat template. `m2.Recipe(answers_only=False)` counts every token instead, which also trains, a little less precisely.

In [8]:
result = job.train()
print(f"{result.steps} steps in {result.seconds:.0f} s; final loss {result.final_loss:.3f}")

meepsquared plan for /home/thinkube/thinkube/mlflow/artifacts/1/3fff0933294740a6ae61724295182e1a/artifacts/model, LoRA, on NVIDIA GB10 (85 GiB free)
  loading      shard by shard to GPU 0 (the GPU shares the system's memory)
  weights      bfloat16 (method lora)
  attention    flash_attention_2 (FlashAttention 2 with packed sequences)
  batches      packed, no padding (the attention (flash_attention_2) keeps the examples apart)
  Liger        kernels and fused loss, without its SwiGLU: the fused LoRA MLP runs instead (from_pretrained's default)
  fused LoRA   on (the model's MLP has a fused kernel)
  compile      layers not compiled (FlashAttention runs outside them)
  checkpoint   0 of 32 layers (one layer for the largest of the 30 micro-batches this job trains, 3,425 tokens (8 examples, end to end); the data's 8 longest would be 7,959: 0.47 GiB keeping everything, 0.03 recomputed whole; its forward and backward 91 / 125 ms in that order; free: 85 of 122 GiB; selective recomputation n

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss


30 steps in 110 s; final loss 1.659


### Reading the loss

The loss measures how surprised the model is by each next token of the training answers, averaged over the step: lower means it predicts the answers better. It falls quickly in the first steps, while the model learns the format and tone of the data, and more slowly after. A loss that stops falling says the model has learned what this data teaches at this learning rate; a loss that climbs says the learning rate is too high. Thirty steps show the first fall; a real fine-tune runs epochs over the data and watches the loss on held-out conversations too.

In [9]:
import pandas as pd

log = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h])[["step", "loss"]]
display(log.set_index("step").T if len(log) <= 40 else log.set_index("step").iloc[::max(1, len(log) // 40)].T)

step,1,2,3,4,5,6,7,8,9,10,...,21,22,23,24,25,26,27,28,29,30
loss,2.103336,1.868346,2.405058,1.95843,2.102154,2.140416,2.171069,1.258845,1.931086,1.617741,...,1.881267,1.815612,2.277692,1.701338,1.728982,1.507832,1.79573,1.4919,1.644944,1.658653


## After training: the same questions

`before` is the answer of the model with its adapter switched off, which is the base model; `after` is the answer of the trained model. The answers move toward the style of the training data. no_robots is written by people answering plainly, so even a short run tends to make the model more direct and less list-like. After a handful of steps the change is small, and it may be none for a question the data says little about: the adapters' effect grows with the data and the steps. A few hundred steps on your own conversations give a model that writes like them.

In [10]:
for q in QUESTIONS:
    with model.disable_adapter():  # the adapter off: the base model, as it was before training
        base = answer(model, tokenizer, q)
    print(f"Q: {q}\nbefore: {base}\nafter:  {answer(model, tokenizer, q)}\n")

Q: Explain in two sentences why the sky is blue.
before: Sunlight appears white, but it is actually made of all the colors of the rainbow, each traveling at a different speed. As this light passes through Earth's atmosphere, shorter blue wavelengths are scattered much more strongly by gas molecules than the longer red wavelengths, filling the sky with blue light that reaches our eyes from every direction.
after:  The sky is blue because of a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it collides with gas molecules and scatters in all directions. Blue light has a shorter wavelength and is scattered more than other colors, making the sky appear blue to our eyes.

Q: Write a haiku about a lighthouse keeper.
before: Watch the waves crash low,
Tending the great spinning light,
Silent in the storm.
after:  The keeper's light shines bright,
Guiding ships through the dark night,
A beacon of hope.



## Save

What LoRA trained is the **adapter**: the small matrices next to each frozen weight, a few tens of MB for an 8B model. `result.save(...)` saves the adapter alone, exactly that. It loads onto the same base model with PEFT, in a notebook or a server, and several adapters can share one copy of the base model.

`SAVE_MERGED` also writes the **merged** model, with `result.save(..., merged=True)`: the base weights with the adapter's change added in, an ordinary 16-bit model that any Transformers or vLLM server loads with no adapter machinery. It is the base model's size. A 4-bit or FP8 base is dequantized for it, so the merged model is a plain BF16 one.

In [11]:
result.save(f"{OUTPUT_DIR}/adapter")
print(f"adapter saved to {OUTPUT_DIR}/adapter")
if SAVE_MERGED:
    result.save(f"{OUTPUT_DIR}/merged", merged=True)
    print(f"merged 16-bit model saved to {OUTPUT_DIR}/merged")

adapter saved to out/01-lora-sft/adapter


## Thinkube only: register the model, so the cluster can serve it

*Elsewhere leave `REGISTER` set to `False`: the cell then only prints a line and does nothing.*

A saved model is a folder; a *served* model answers requests. On Thinkube, `REGISTER = True` takes the model from one to the other. It merges the adapter into the base weights, uploads the result to the training run in Thinkube Experiments, registers a model version linked to that run (so the served model points back to its data and settings), and writes a catalogue entry to your private metadata repository through Thinkube Control. The model then appears in the LLM Gateway as deployable, and `load_llm_model` serves it on vLLM. The upload of the merged weights takes some minutes.

The catalogue entry says how to serve the model: its size, its context length, whether it thinks, whether it calls tools. The one below is filled from the base model's; change `FINETUNED_NAME` and the description to yours.

In [12]:
if REGISTER:
    if MODEL_SOURCE != "thinkube":
        raise RuntimeError("REGISTER needs Thinkube: the model is registered in the cluster's model registry and catalogue")
    import thinkube_models as tkm

    CATALOG_ENTRY = {
        "id": FINETUNED_NAME, "name": FINETUNED_NAME, "params_b": 8.0, "active_params_b": None,
        "quantization": "BF16", "context_length": 131072,
        "description": 'A chat model fine-tuned with meepsquared on no_robots conversations (notebook 01).',
        "server_type": ["vllm"], "task": "text-generation", "reasoning_format": None, "tool_use": True,
        "stop_tokens": [], "license": "apache-2.0", "gated": False, "serving_name": FINETUNED_NAME, "is_finetuned": True,
    }
    tkm.register_finetuned_model(model, tokenizer, CATALOG_ENTRY, RUN_ID)
else:
    print("REGISTER is False: the adapter is saved above; set REGISTER = True on Thinkube to serve the merged model")

if RUN_ID:
    mlflow.end_run()

REGISTER is False: the adapter is saved above; set REGISTER = True on Thinkube to serve the merged model
🏃 View run 01-lora-Qwen3.5-9B at: http://mlflow.mlflow.svc.cluster.local:5000/#/experiments/6/runs/86ba073a98db4a99836d71c1a3b4608a
🧪 View experiment at: http://mlflow.mlflow.svc.cluster.local:5000/#/experiments/6


## Where to go next

* [02 · Frozen weights in FP8](02-fp8-weights.ipynb): the same training with the base model's weights in FP8, faster and smaller on a GPU with FP8 tensor cores.
* [03 · LoRA on every expert of a mixture of experts](03-moe-experts.ipynb).
* [Training jobs](https://thinkube.org/meepsquared/training-job.html) in the documentation: every field of a job, every default of auto mode and its source.
* The [documentation](https://thinkube.org/meepsquared/): trainer settings, saving, several GPUs, and the [benchmarks](https://thinkube.org/meepsquared/benchmarks.html) behind the numbers.